# EXP4 Inference Prompt Mix

This notebook builds two independent prompt-mixing runs for `multi_inc16`:

1. `all_prompts`: max-over-prompts mix using the 10 original prompts.
2. `exclude_p1_p3`: max-over-prompts mix excluding prompts 1 and 3.

The notebook is structured so the two sections do not mix results:
each run has its own configuration object, output directory, saved volumes,
evaluation dataframe, and macro summary.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

from paths import EXP4_RESULTS_DIR, EXP4_INFERENCE_DIR, EXP4_COORDS_DIR
from experiments.config import EXP4_VAL_TOMOS, EXP4_NUM_PROMPTS

print(f"Project root: {PROJECT_ROOT}")
print(f"EXP4 results dir: {EXP4_RESULTS_DIR}")
print(f"EXP4 inference dir: {EXP4_INFERENCE_DIR}")
print(f"GT coords dir: {EXP4_COORDS_DIR}")
print(f"Validation tomograms ({len(EXP4_VAL_TOMOS)}): {EXP4_VAL_TOMOS}")
print(f"Expected prompt count: {EXP4_NUM_PROMPTS}")


Project root: /home/carloshg/Dev/cryoet-particle-picking
EXP4 results dir: /home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations
EXP4 inference dir: /home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations/inference
GT coords dir: /home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations/coords
Validation tomograms (5): ['tomo_rec_5_snr1.66', 'tomo_rec_6_snr1.17', 'tomo_rec_7_snr1.13', 'tomo_rec_8_snr0.57', 'tomo_rec_9_snr1.28']
Expected prompt count: 10


In [2]:
MIX_ROOT_DIR = EXP4_RESULTS_DIR / "inference_mix"
MIX_ROOT_DIR.mkdir(parents=True, exist_ok=True)


def build_mix_config(
    scenario_name: str,
    prompt_indices,
    checkpoint_type: str = "multi",
    increment: int = 16,
) -> dict:
    prompt_indices = list(prompt_indices)
    run_name = (
        f"{checkpoint_type}_inc{increment}_max_over_{scenario_name}"
        if checkpoint_type != "base"
        else f"base_max_over_{scenario_name}"
    )
    out_dir = MIX_ROOT_DIR / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    return {
        "scenario_name": scenario_name,
        "checkpoint_type": checkpoint_type,
        "increment": increment,
        "prompt_indices": prompt_indices,
        "out_dir": out_dir,
        "run_name": run_name,
    }


def inference_result_dir(config: dict, prompt_idx: int) -> Path:
    if config["checkpoint_type"] == "base":
        return EXP4_INFERENCE_DIR / f"base_prompt{prompt_idx}"
    return (
        EXP4_INFERENCE_DIR
        / f"{config['checkpoint_type']}_inc{config['increment']}_prompt{prompt_idx}"
    )


def prediction_pt_path(config: dict, prompt_idx: int, tomo_name: str) -> Path:
    return (
        inference_result_dir(config, prompt_idx)
        / "full_segmentation_output"
        / f"{tomo_name}.pt"
    )


def validate_prompt_predictions(config: dict) -> None:
    missing = []
    for tomo_name in EXP4_VAL_TOMOS:
        for prompt_idx in config["prompt_indices"]:
            pt_path = prediction_pt_path(config, prompt_idx, tomo_name)
            if not pt_path.exists():
                missing.append(str(pt_path))

    if missing:
        print("Missing input .pt files:")
        for path in missing[:20]:
            print("  ", path)
        if len(missing) > 20:
            print(f"  ... and {len(missing) - 20} more")
        raise FileNotFoundError(
            f"Cannot continue for {config['scenario_name']}: missing {len(missing)} files"
        )

    print(
        f"All expected prediction volumes are present for {config['scenario_name']}."
    )


def mix_predictions(config: dict):
    saved_files = []
    for tomo_name in EXP4_VAL_TOMOS:
        preds = []
        for prompt_idx in config["prompt_indices"]:
            pt_path = prediction_pt_path(config, prompt_idx, tomo_name)
            pred = torch.load(pt_path, map_location="cpu")
            if not isinstance(pred, torch.Tensor):
                pred = torch.tensor(pred)
            preds.append(pred.float())

        stacked = torch.stack(preds, dim=0)
        mixed_pred = torch.max(stacked, dim=0).values

        out_path = config["out_dir"] / f"{tomo_name}.pt"
        torch.save(mixed_pred, out_path)
        saved_files.append(out_path)

        print(
            f"  {config['scenario_name']} | {tomo_name}: "
            f"saved {out_path.name} | shape={tuple(mixed_pred.shape)}"
        )

    return saved_files


def summarize_saved_files(config: dict, saved_files) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "scenario": config["scenario_name"],
            "tomo_name": [path.stem for path in saved_files],
            "pt_path": [str(path) for path in saved_files],
        }
    )
    display(summary)
    return summary


def load_gt_coords(tomo_name: str) -> np.ndarray:
    coords_file = EXP4_COORDS_DIR / f"{tomo_name}_thyroglobulin_coords.csv"
    if not coords_file.exists():
        return np.empty((0, 3), dtype=float)
    df = pd.read_csv(coords_file)
    return df[["X", "Y", "Z"]].to_numpy(dtype=float)


def compute_metrics(
    pred_coords: np.ndarray,
    gt_coords: np.ndarray,
    distance_threshold: float = 15.0,
) -> dict:
    if len(pred_coords) == 0 and len(gt_coords) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "tp": 0, "fp": 0, "fn": 0}
    if len(pred_coords) == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "tp": 0,
            "fp": 0,
            "fn": int(len(gt_coords)),
        }
    if len(gt_coords) == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "tp": 0,
            "fp": int(len(pred_coords)),
            "fn": 0,
        }

    distances = cdist(pred_coords, gt_coords)
    matched_gt = set()
    tp = 0

    for i in range(len(pred_coords)):
        min_dist = float("inf")
        best_j = -1
        for j in range(len(gt_coords)):
            if j not in matched_gt and distances[i, j] < min_dist:
                min_dist = distances[i, j]
                best_j = j

        if best_j != -1 and min_dist <= distance_threshold:
            tp += 1
            matched_gt.add(best_j)

    fp = int(len(pred_coords) - tp)
    fn = int(len(gt_coords) - tp)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
    }


def tensor_to_3d_numpy(pred_tensor) -> np.ndarray:
    if isinstance(pred_tensor, torch.Tensor):
        arr = pred_tensor.detach().cpu().float().numpy()
    else:
        arr = np.asarray(pred_tensor, dtype=np.float32)

    arr = np.squeeze(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume after squeeze, got shape={arr.shape}")
    return arr.astype(np.float32)


def extract_pred_coords_from_mixed_pt(
    pt_path: Path,
    seg_thresh: float = 0.5,
    min_component_voxels: int = 5,
) -> np.ndarray:
    pred = torch.load(pt_path, map_location="cpu")
    vol = tensor_to_3d_numpy(pred)
    mask = vol >= seg_thresh
    labeled, n_comp = ndi.label(mask)

    if n_comp == 0:
        return np.empty((0, 3), dtype=float)

    sizes = np.bincount(labeled.ravel())
    valid_labels = np.where(sizes >= min_component_voxels)[0]
    valid_labels = valid_labels[valid_labels != 0]

    if len(valid_labels) == 0:
        return np.empty((0, 3), dtype=float)

    centers_zyx = ndi.center_of_mass(mask, labeled, valid_labels)
    coords_xyz = np.array([[c[2], c[1], c[0]] for c in centers_zyx], dtype=float)
    return coords_xyz


def evaluate_mixed_predictions(
    config: dict,
    seg_thresh: float = 0.5,
    min_component_voxels: int = 5,
    distance_threshold: float = 15.0,
) -> pd.DataFrame:
    rows = []
    for tomo_name in EXP4_VAL_TOMOS:
        mixed_pt = config["out_dir"] / f"{tomo_name}.pt"
        if not mixed_pt.exists():
            print(f"Warning: missing mixed prediction for {tomo_name}: {mixed_pt}")
            continue

        pred_coords = extract_pred_coords_from_mixed_pt(
            mixed_pt,
            seg_thresh=seg_thresh,
            min_component_voxels=min_component_voxels,
        )
        gt_coords = load_gt_coords(tomo_name)
        metrics = compute_metrics(
            pred_coords,
            gt_coords,
            distance_threshold=distance_threshold,
        )
        rows.append(
            {
                "scenario": config["scenario_name"],
                "tomo_name": tomo_name,
                "n_pred": int(len(pred_coords)),
                "n_gt": int(len(gt_coords)),
                **metrics,
            }
        )

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values("tomo_name").reset_index(drop=True)


def summarize_macro(df_metrics: pd.DataFrame) -> pd.DataFrame:
    if len(df_metrics) == 0:
        return pd.DataFrame()
    return (
        df_metrics[["precision", "recall", "f1", "n_pred", "n_gt"]]
        .mean()
        .to_frame(name="mean")
        .T
        .round(4)
    )


## Section 1: All Prompts

Mix the 10 original prompt predictions for `multi_inc16` using voxelwise max and evaluate the resulting volumes.


In [3]:
config_all_prompts = build_mix_config(
    scenario_name="all_prompts",
    prompt_indices=range(EXP4_NUM_PROMPTS),
    checkpoint_type="multi",
    increment=16,
)

print(config_all_prompts)
validate_prompt_predictions(config_all_prompts)
saved_files_all_prompts = mix_predictions(config_all_prompts)
df_saved_all_prompts = summarize_saved_files(
    config_all_prompts,
    saved_files_all_prompts,
)

df_mix_metrics_all_prompts = evaluate_mixed_predictions(config_all_prompts)
display(df_mix_metrics_all_prompts.round(4))

df_mix_macro_all_prompts = summarize_macro(df_mix_metrics_all_prompts)
display(df_mix_macro_all_prompts)


{'scenario_name': 'all_prompts', 'checkpoint_type': 'multi', 'increment': 16, 'prompt_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'out_dir': PosixPath('/home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations/inference_mix/multi_inc16_max_over_all_prompts'), 'run_name': 'multi_inc16_max_over_all_prompts'}
All expected prediction volumes are present for all_prompts.


/tmp/ipykernel_17150/459250383.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pred = torch.load(pt_path, map_location="cpu")


  all_prompts | tomo_rec_5_snr1.66: saved tomo_rec_5_snr1.66.pt | shape=(196, 642, 642)
  all_prompts | tomo_rec_6_snr1.17: saved tomo_rec_6_snr1.17.pt | shape=(196, 642, 642)
  all_prompts | tomo_rec_7_snr1.13: saved tomo_rec_7_snr1.13.pt | shape=(196, 642, 642)
  all_prompts | tomo_rec_8_snr0.57: saved tomo_rec_8_snr0.57.pt | shape=(196, 642, 642)
  all_prompts | tomo_rec_9_snr1.28: saved tomo_rec_9_snr1.28.pt | shape=(196, 642, 642)


,scenario,tomo_name,pt_path
0,all_prompts,tomo_rec_5_snr1.66,/home/carloshg/Dev/cryoet-particle-picking/res...
1,all_prompts,tomo_rec_6_snr1.17,/home/carloshg/Dev/cryoet-particle-picking/res...
2,all_prompts,tomo_rec_7_snr1.13,/home/carloshg/Dev/cryoet-particle-picking/res...
3,all_prompts,tomo_rec_8_snr0.57,/home/carloshg/Dev/cryoet-particle-picking/res...
4,all_prompts,tomo_rec_9_snr1.28,/home/carloshg/Dev/cryoet-particle-picking/res...


/tmp/ipykernel_17150/459250383.py:194: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pred = torch.load(pt_path, map_location="cpu")
/tmp/ipykernel_17150/459250383.py:194: Fu

,scenario,tomo_name,n_pred,n_gt,precision,recall,f1,tp,fp,fn
0,all_prompts,tomo_rec_5_snr1.66,167,127,0.7545,0.9921,0.8571,126,41,1
1,all_prompts,tomo_rec_6_snr1.17,159,120,0.7547,1.0000,0.8602,120,39,0
2,all_prompts,tomo_rec_7_snr1.13,173,133,0.7572,0.9850,0.8562,131,42,2
3,all_prompts,tomo_rec_8_snr0.57,229,115,0.4760,0.9478,0.6337,109,120,6
4,all_prompts,tomo_rec_9_snr1.28,158,127,0.7911,0.9843,0.8772,125,33,2


,precision,recall,f1,n_pred,n_gt
mean,0.7067,0.9818,0.8169,177.2,124.4


## Section 2: Excluding Prompts 1 and 3

Repeat the exact same workflow, but exclude prompts 1 and 3 from the voxelwise max mix.


In [4]:
config_excl_p1_p3 = build_mix_config(
    scenario_name="exclude_p1_p3",
    prompt_indices=[0, 2, 4, 5, 6, 7, 8, 9],
    checkpoint_type="multi",
    increment=16,
)

print(config_excl_p1_p3)
validate_prompt_predictions(config_excl_p1_p3)
saved_files_excl_p1_p3 = mix_predictions(config_excl_p1_p3)
df_saved_excl_p1_p3 = summarize_saved_files(
    config_excl_p1_p3,
    saved_files_excl_p1_p3,
)

df_mix_metrics_excl_p1_p3 = evaluate_mixed_predictions(config_excl_p1_p3)
display(df_mix_metrics_excl_p1_p3.round(4))

df_mix_macro_excl_p1_p3 = summarize_macro(df_mix_metrics_excl_p1_p3)
display(df_mix_macro_excl_p1_p3)


{'scenario_name': 'exclude_p1_p3', 'checkpoint_type': 'multi', 'increment': 16, 'prompt_indices': [0, 2, 4, 5, 6, 7, 8, 9], 'out_dir': PosixPath('/home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations/inference_mix/multi_inc16_max_over_exclude_p1_p3'), 'run_name': 'multi_inc16_max_over_exclude_p1_p3'}
All expected prediction volumes are present for exclude_p1_p3.


/tmp/ipykernel_17150/459250383.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pred = torch.load(pt_path, map_location="cpu")


  exclude_p1_p3 | tomo_rec_5_snr1.66: saved tomo_rec_5_snr1.66.pt | shape=(196, 642, 642)
  exclude_p1_p3 | tomo_rec_6_snr1.17: saved tomo_rec_6_snr1.17.pt | shape=(196, 642, 642)
  exclude_p1_p3 | tomo_rec_7_snr1.13: saved tomo_rec_7_snr1.13.pt | shape=(196, 642, 642)
  exclude_p1_p3 | tomo_rec_8_snr0.57: saved tomo_rec_8_snr0.57.pt | shape=(196, 642, 642)
  exclude_p1_p3 | tomo_rec_9_snr1.28: saved tomo_rec_9_snr1.28.pt | shape=(196, 642, 642)


,scenario,tomo_name,pt_path
0,exclude_p1_p3,tomo_rec_5_snr1.66,/home/carloshg/Dev/cryoet-particle-picking/res...
1,exclude_p1_p3,tomo_rec_6_snr1.17,/home/carloshg/Dev/cryoet-particle-picking/res...
2,exclude_p1_p3,tomo_rec_7_snr1.13,/home/carloshg/Dev/cryoet-particle-picking/res...
3,exclude_p1_p3,tomo_rec_8_snr0.57,/home/carloshg/Dev/cryoet-particle-picking/res...
4,exclude_p1_p3,tomo_rec_9_snr1.28,/home/carloshg/Dev/cryoet-particle-picking/res...


/tmp/ipykernel_17150/459250383.py:194: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pred = torch.load(pt_path, map_location="cpu")
/tmp/ipykernel_17150/459250383.py:194: Fu

,scenario,tomo_name,n_pred,n_gt,precision,recall,f1,tp,fp,fn
0,exclude_p1_p3,tomo_rec_5_snr1.66,167,127,0.7545,0.9921,0.8571,126,41,1
1,exclude_p1_p3,tomo_rec_6_snr1.17,159,120,0.7547,1.0000,0.8602,120,39,0
2,exclude_p1_p3,tomo_rec_7_snr1.13,173,133,0.7572,0.9850,0.8562,131,42,2
3,exclude_p1_p3,tomo_rec_8_snr0.57,229,115,0.4760,0.9478,0.6337,109,120,6
4,exclude_p1_p3,tomo_rec_9_snr1.28,158,127,0.7911,0.9843,0.8772,125,33,2


,precision,recall,f1,n_pred,n_gt
mean,0.7067,0.9818,0.8169,177.2,124.4


## Final Comparison

Compare the two isolated runs without reusing mutable output variables.


In [5]:
comparison_tables = []

if "df_mix_metrics_all_prompts" in globals() and len(df_mix_metrics_all_prompts) > 0:
    comparison_tables.append(df_mix_metrics_all_prompts.copy())
else:
    print("Warning: df_mix_metrics_all_prompts is not available.")

if "df_mix_metrics_excl_p1_p3" in globals() and len(df_mix_metrics_excl_p1_p3) > 0:
    comparison_tables.append(df_mix_metrics_excl_p1_p3.copy())
else:
    print("Warning: df_mix_metrics_excl_p1_p3 is not available.")

if comparison_tables:
    df_metrics_comparison = pd.concat(comparison_tables, ignore_index=True)
    display(df_metrics_comparison.round(4))

    df_macro_comparison = (
        df_metrics_comparison.groupby("scenario")[
            ["precision", "recall", "f1", "n_pred", "n_gt"]
        ]
        .mean()
        .reset_index()
        .round(4)
    )
    display(df_macro_comparison)

    df_macro_pivot = df_macro_comparison.set_index("scenario")
    if {"all_prompts", "exclude_p1_p3"}.issubset(df_macro_pivot.index):
        delta = (
            df_macro_pivot.loc["exclude_p1_p3"]
            - df_macro_pivot.loc["all_prompts"]
        ).to_frame(name="exclude_p1_p3_minus_all_prompts")
        display(delta.round(4))
else:
    print("No metrics tables available for comparison.")


,scenario,tomo_name,n_pred,n_gt,precision,recall,f1,tp,fp,fn
0,all_prompts,tomo_rec_5_snr1.66,167,127,0.7545,0.9921,0.8571,126,41,1
1,all_prompts,tomo_rec_6_snr1.17,159,120,0.7547,1.0000,0.8602,120,39,0
2,all_prompts,tomo_rec_7_snr1.13,173,133,0.7572,0.9850,0.8562,131,42,2
3,all_prompts,tomo_rec_8_snr0.57,229,115,0.4760,0.9478,0.6337,109,120,6
4,all_prompts,tomo_rec_9_snr1.28,158,127,0.7911,0.9843,0.8772,125,33,2
5,exclude_p1_p3,tomo_rec_5_snr1.66,167,127,0.7545,0.9921,0.8571,126,41,1
6,exclude_p1_p3,tomo_rec_6_snr1.17,159,120,0.7547,1.0000,0.8602,120,39,0
7,exclude_p1_p3,tomo_rec_7_snr1.13,173,133,0.7572,0.9850,0.8562,131,42,2
8,exclude_p1_p3,tomo_rec_8_snr0.57,229,115,0.4760,0.9478,0.6337,109,120,6
9,exclude_p1_p3,tomo_rec_9_snr1.28,158,127,0.7911,0.9843,0.8772,125,33,2


,scenario,precision,recall,f1,n_pred,n_gt
0,all_prompts,0.7067,0.9818,0.8169,177.2,124.4
1,exclude_p1_p3,0.7067,0.9818,0.8169,177.2,124.4


,exclude_p1_p3_minus_all_prompts
precision,0.0
recall,0.0
f1,0.0
n_pred,0.0
n_gt,0.0
